# FitForge Diet Recommender — Colab Training

Trains a compact dish-suitability model and exports it to **TFLite** for fully offline on-device inference in the Flutter app.

The catalog is now **multi-country** (India + USA + UK + Australia + Canada, ~260 dishes). The model stays country-agnostic: the app hard-filters candidate dishes by the user's country *before* scoring, so country is **not** a model feature and the 16-feature spec is unchanged. To retrain on the latest dishes, just re-run this notebook with the current `diet_catalog.json` — it auto-picks up every dish, including the new countries.

**How to run**
1. Runtime -> Change runtime type -> **GPU**.
2. Run every cell top to bottom.
3. When prompted, upload `assets/database/diet_catalog.json` from the repo.
4. At the end, two files download: `diet_recommender.tflite` and `feature_scaler.json`.
5. Put both in the app at `assets/models/` (exact names). The app auto-detects them.

**Critical:** the 16-feature order/formulas in `build_features` below MUST match `buildDishFeatures()` in `lib/services/diet_recommender_model.dart`. If you change one, change both.

In [ ]:
import json, numpy as np, tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

# Upload diet_catalog.json (from assets/database/ in the repo)
from google.colab import files
uploaded = files.upload()
catalog_name = next(k for k in uploaded if k.endswith('.json'))
catalog = json.loads(uploaded[catalog_name].decode('utf-8'))
dishes = catalog['dishes']
print(f'Loaded {len(dishes)} dishes from {catalog_name}')

In [ ]:
# ---- Feature spec (MUST mirror buildDishFeatures in the Dart app) ----
FEATURE_DIM = 16
GOALS = ['fat_loss', 'muscle_gain', 'endurance', 'maintenance']

def build_features(d, goal, zone_match):
    kcal = max(float(d['kcal']), 1.0)
    protein, carbs, fat, fiber = float(d['protein']), float(d['carbs']), float(d['fat']), float(d['fiber'])
    return [
        d['kcal'] / 700.0,
        protein / 50.0,
        carbs / 120.0,
        fat / 45.0,
        fiber / 15.0,
        (protein * 4.0 / kcal * 100.0) / 40.0,
        fat * 9.0 / kcal,
        carbs * 4.0 / kcal,
        1.0 if d['diet'] == 'veg' else 0.0,
        1.0 if d['diet'] == 'egg' else 0.0,
        1.0 if d['diet'] == 'nonveg' else 0.0,
        float(zone_match),
        1.0 if goal == 'fat_loss' else 0.0,
        1.0 if goal == 'muscle_gain' else 0.0,
        1.0 if goal == 'endurance' else 0.0,
        1.0 if goal == 'maintenance' else 0.0,
    ]

def clamp01(x):
    return max(0.0, min(1.0, x))

# ---- Label: gym/celebrity-plan informed suitability (richer than the Dart heuristic) ----
def suitability_label(d, goal, zone_match):
    kcal = max(float(d['kcal']), 1.0)
    protein, carbs, fat, fiber = float(d['protein']), float(d['carbs']), float(d['fat']), float(d['fiber'])
    protein_density = clamp01((protein * 4.0 / kcal * 100.0) / 35.0)
    abs_protein = clamp01(protein / 35.0)
    fiber_score = clamp01(fiber / 12.0)
    fat_pct = fat * 9.0 / kcal
    lean_score = clamp01(1.0 - fat_pct / 0.45)
    carb_pct = carbs * 4.0 / kcal
    carb_score = clamp01(carb_pct / 0.6)
    if goal == 'fat_loss':
        base = 0.5 * protein_density + 0.3 * fiber_score + 0.2 * lean_score
    elif goal == 'muscle_gain':
        base = 0.55 * protein_density + 0.3 * abs_protein + 0.15 * carb_score
    elif goal == 'endurance':
        base = 0.5 * carb_score + 0.3 * protein_density + 0.2 * fiber_score
    else:  # maintenance
        base = 0.4 * protein_density + 0.3 * fiber_score + 0.3 * lean_score
    # Region familiarity nudge + small noise so the net learns a smooth surface
    base += 0.05 * zone_match
    base += np.random.normal(0.0, 0.04)
    return clamp01(base)

# ---- Synthesize the training set ----
np.random.seed(42)
REPLICAS = 24  # noisy samples per (dish, goal, zone_match)
X, y = [], []
for d in dishes:
    for goal in GOALS:
        for zone_match in (0.0, 1.0):
            for _ in range(REPLICAS):
                X.append(build_features(d, goal, zone_match))
                y.append(suitability_label(d, goal, zone_match))
X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)
print('Dataset:', X.shape, y.shape)

In [ ]:
# ---- KMeans calorie-banding (insight from soumillll/Diet-Recommendation-System) ----
# We surface the bands for analysis/labeling sanity. They are NOT added to the model
# features (that would break parity with the Dart feature vector).
kcals = np.array([[float(d['kcal'])] for d in dishes])
bands = KMeans(n_clusters=3, n_init=10, random_state=42).fit(kcals)
order = np.argsort(bands.cluster_centers_.flatten())
label_for = {order[0]: 'low', order[1]: 'medium', order[2]: 'high'}
for d, c in zip(dishes, bands.labels_):
    d['calorie_band'] = label_for[c]
from collections import Counter
print('Calorie bands:', Counter(d['calorie_band'] for d in dishes))

In [ ]:
# ---- Standardize + train the MLP ----
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X).astype(np.float32)
X_tr, X_te, y_tr, y_te = train_test_split(Xs, y, test_size=0.15, random_state=42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(FEATURE_DIM,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model.fit(X_tr, y_tr, validation_data=(X_te, y_te), epochs=40, batch_size=256, verbose=2)
print('Test MAE:', model.evaluate(X_te, y_te, verbose=0)[1])

In [ ]:
# ---- Export TFLite model + feature scaler, then download ----
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open('diet_recommender.tflite', 'wb') as f:
    f.write(tflite_model)

scaler_json = {
    'mean': [float(v) for v in scaler.mean_],
    'std': [float(v) for v in scaler.scale_],
    'feature_dim': FEATURE_DIM,
}
with open('feature_scaler.json', 'w') as f:
    json.dump(scaler_json, f, indent=2)

print('Wrote diet_recommender.tflite (%d bytes) and feature_scaler.json' % len(tflite_model))

# Quick sanity check: a high-protein dish should beat a high-fat one for muscle_gain
def predict(d, goal, zone_match=1.0):
    f = np.array([build_features(d, goal, zone_match)], dtype=np.float32)
    return float(model.predict(scaler.transform(f), verbose=0)[0][0])
best = sorted(dishes, key=lambda d: predict(d, 'muscle_gain'), reverse=True)[:5]
print('Top muscle-gain dishes:', [d['name'] for d in best])

from google.colab import files
files.download('diet_recommender.tflite')
files.download('feature_scaler.json')